# BEBOP1 Workshop Tutorial
## From Geometry Optimization to Bond Energy Analysis with PySCF + MinPop

This notebook walks through a complete computational chemistry workflow:

| Step | Description | Method |
|------|-------------|--------|
| 1 | Geometry optimization of CH₃OH | B3LYP/STO-3G (PySCF) |
| 2 | MinPop single-point calculation | HF/6-31G* |
| 3 | MinPop single-point calculation | HF/cc-pVTZ |
| 4 | BEBOP1 bond energy calculation | From MinPop data |

**References:**
- MinPop: https://github.com/keithgroup/MinPop
- BEBOP1: https://github.com/keithgroup/BEBOP1_v2.0.0
- Zulueta et al., *J. Chem. Theory Comput.* **2022**, *18*, 4774–4794. [DOI: 10.1021/acs.jctc.2c00334](https://doi.org/10.1021/acs.jctc.2c00334)


## 0. Install Dependencies

Run this cell first to install PySCF, the geometry optimizer, and clone the BEBOP1 code.
This takes about 1–2 minutes on Colab.


In [1]:
# Install packages (run this first!)
!pip install pyscf geometric scipy numpy -q

# Clone and install BEBOP1
!git clone https://github.com/keithgroup/BEBOP1_v2.0.0.git 2>/dev/null || echo "Already cloned"
!cd BEBOP1_v2.0.0 && pip install . -q

# Clone MinPop reference (for comparison)
!git clone https://github.com/keithgroup/MinPop.git 2>/dev/null || echo "Already cloned"

print("\n✅ All dependencies installed!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.0/386.0 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 MB 14.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

✅ All dependencies installed!


## 0b. Imports

In [2]:
import numpy as np
import pyscf
import pyscf.lo
import pyscf.gto
import scipy.linalg
import sys, os

# Add BEBOP1 to path
sys.path.insert(0, 'BEBOP1_v2.0.0')
import bebop1_equation as bebop_eq
import spatial_geom as sg

BOHR_TO_ANG = 0.529177249  # Bohr -> Angstrom

print(f"PySCF version: {pyscf.__version__}")
print("Ready to go!")


PySCF version: 2.12.1
Ready to go!


## Step 1: Geometry Optimization of CH₃OH

We start with a rough initial geometry of methanol and optimize it using
**B3LYP/STO-3G**. This is a minimal basis DFT calculation — fast enough to
run live in a workshop, but still gives a reasonable equilibrium structure.

PySCF uses the [geomeTRIC](https://github.com/leeping/geomeTRIC) optimizer
under the hood, which works with redundant internal coordinates.


In [3]:
# Build CH3OH with an initial (approximate) geometry
mol = pyscf.gto.M(
    atom='''
    C   0.0000   0.0000   0.0000
    O   1.4300   0.0000   0.0000
    H  -0.3900   1.0100   0.0000
    H  -0.3900  -0.5050   0.8750
    H  -0.3900  -0.5050  -0.8750
    H   1.8100   0.9400   0.0000
    ''',
    basis='sto-3g',
    charge=0,
    spin=0,  # singlet (all electrons paired)
)

print(f"Molecule:  CH3OH (methanol)")
print(f"Basis:     STO-3G")
print(f"# atoms:   {mol.natm}")
print(f"# basis functions: {mol.nao}")


Molecule:  CH3OH (methanol)
Basis:     STO-3G
# atoms:   6
# basis functions: 14


In [4]:
from pyscf import dft
from pyscf.geomopt.geometric_solver import optimize

# Set up B3LYP calculation
mf = dft.RKS(mol)
mf.xc = 'B3LYP'

# Compute initial energy
e_init = mf.kernel()
print(f"Initial B3LYP/STO-3G energy: {e_init:.10f} Hartree")


converged SCF energy = -114.168325962507
Initial B3LYP/STO-3G energy: -114.1683259625 Hartree


In [15]:
# Run the geometry optimization (takes ~10-15 seconds)
mol_eq = optimize(mf, maxsteps=100)

# Display the optimized geometry
# NOTE: mol_eq from geomeTRIC returns coordinates in BOHR internally
atom_symbols = [mol_eq.atom_symbol(i) for i in range(mol_eq.natm)]
opt_coords_ang = mol_eq.atom_coords() * BOHR_TO_ANG

print(f"\nOptimized geometry (Angstrom):")
print(f"  {'Atom':<6} {'X':>12} {'Y':>12} {'Z':>12}")
print(f"  {'-'*44}")
for i, sym in enumerate(atom_symbols):
    x, y, z = opt_coords_ang[i]
    print(f"  {sym:<6} {x:12.6f} {y:12.6f} {z:12.6f}")

# Compute optimized energy
mf_opt = dft.RKS(mol_eq)
mf_opt.xc = 'B3LYPG'
e_opt = mf_opt.kernel()
print(f"\nOptimized B3LYP/STO-3G energy: {e_opt:.10f} Hartree")
print(f"Energy lowering: {(e_opt - e_init)*627.509:.2f} kcal/mol")


geometric-optimize called with the following command line:
/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-edc59e52-f7b9-4ee6-ac87-61f0120c625c.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **              ********    


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   O   1.430000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -0.390000   1.010000   0.000000    0.000000  0.000000  0.000000
   H  -0.390000  -0.505000   0.875000    0.000000  0.000000  0.000000
   H  -0.390000  -0.505000  -0.875000    0.000000  0.000000  0.000000
   H   1.810000   0.940000   0.000000   -0.000000 -0.000000  0.000000

WARN: Mole.unit (angstrom) is changed to Bohr

converged SCF energy = -114.168325962482
--------------- RKS_Scanner gradients ---------------
         x                y                z
0 C    -0.0036765143    -0.0288890475     0.0000000000
1 O    -0.0443990106     0.0370607571    -0.0000000000
2 H     0.0148482856    -0.0177131922    -0.0000000000
3 H     0.0074375556     0.0119199536    -0.0194379347
4 H     0.0074375556     0.0119199536     0.

Step    0 : Gradient = 3.271e-02/5.783e-02 (rms/max) Energy = -114.1683259625
Hessian Eigenvalues: 2.30000e-02 5.00000e-02 5.00000e-02 ... 3.56722e-01 4.09893e-01 4.53251e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   0.033924   0.024488  -0.000000    0.033924  0.024488 -0.000000
   O   1.494388  -0.071805   0.000000    0.064388 -0.071805  0.000000
   H  -0.388483   1.056519  -0.000000    0.001517  0.046519 -0.000000
   H  -0.374316  -0.500979   0.899017    0.015684  0.004021  0.024017
   H  -0.374316  -0.500979  -0.899017    0.015684  0.004021 -0.024017
   H   1.679432   0.932644   0.000000   -0.130568 -0.007356  0.000000
converged SCF energy = -114.17361527939
--------------- RKS_Scanner gradients ---------------
         x                y                z
0 C     0.0221954034     0.0152863065    -0.0000000000
1 O     0.0000214676    -0.0126354173     0.0000000000
2 H    -0.0058631247     0.0039475115    -0.0000000000
3 H     0.0000825101    -0.0023936595     0.0039981851
4 H     0.0000825101    -0.0023936595    -0.0039981851
5 H    -0.0165154283    -0.0018106087 

Step    1 : Displace = 7.303e-02/1.309e-01 (rms/max) Trust = 1.000e-01 (=) Grad = 1.446e-02/2.695e-02 (rms/max) E (change) = -114.1736152794 (-5.289e-03) Quality = 0.548
Hessian Eigenvalues: 2.30000e-02 5.00000e-02 5.00000e-02 ... 3.57069e-01 4.20345e-01 4.53620e-01


...


Geometry optimization cycle 5
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   0.012294   0.013586  -0.000000    0.000297  0.001129  0.000000
   O   1.487185  -0.047539  -0.000000    0.002436 -0.000740 -0.000000
   H  -0.392180   1.044327  -0.000000   -0.002972  0.000441  0.000000
   H  -0.388876  -0.508184   0.895668   -0.000067 -0.000018 -0.000002
   H  -0.388876  -0.508184  -0.895668   -0.000067 -0.000018  0.000002
   H   1.740746   0.946338  -0.000000    0.000223 -0.000541 -0.000000
converged SCF energy = -114.175513087861
--------------- RKS_Scanner gradients ---------------
         x                y                z
0 C     0.0003245194     0.0001137830     0.0000000000
1 O    -0.0001080105     0.0000149379    -0.0000000000
2 H    -0.0000861960    -0.0000394113     0.0000000000
3 H    -0.0000310767    -0.0000005811    -0.0000188818
4 H    -0.0000310767    -0.0000005811     0.0000188818
5 H    -0.0000652334    -0.0000923424

Step    4 : Displace = 1.691e-03/2.907e-03 (rms/max) Trust = 2.000e-01 (+) Grad = 1.605e-04/3.439e-04 (rms/max) E (change) = -114.1755130879 (-7.779e-06) Quality = 0.965
Hessian Eigenvalues: 2.30000e-02 4.99900e-02 5.00000e-02 ... 3.56830e-01 4.45030e-01 4.74222e-01



Geometry optimization cycle 6
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   0.012017   0.013442   0.000000   -0.000277 -0.000144  0.000000
   O   1.487176  -0.047359   0.000000   -0.000009  0.000180  0.000000
   H  -0.391894   1.044432  -0.000000    0.000286  0.000104  0.000000
   H  -0.388934  -0.508279   0.895825   -0.000058 -0.000094  0.000157
   H  -0.388934  -0.508279  -0.895825   -0.000058 -0.000094 -0.000157
   H   1.740677   0.946651   0.000000   -0.000069  0.000313  0.000000
converged SCF energy = -114.175513249212
--------------- RKS_Scanner gradients ---------------
         x                y                z
0 C     0.0000858371     0.0000005820    -0.0000000000
1 O    -0.0000307334     0.0000006574    -0.0000000000
2 H    -0.0000129809     0.0000192948     0.0000000000
3 H    -0.0000189857    -0.0000108432     0.0000216929
4 H    -0.0000189857    -0.0000108432    -0.0000216929
5 H    -0.0000011998    -0.0000030383

Step    5 : Displace = 2.438e-04/3.838e-04 (rms/max) Trust = 2.828e-01 (+) Grad = 4.235e-05/8.584e-05 (rms/max) E (change) = -114.1755132492 (-1.614e-07) Quality = 1.080
Hessian Eigenvalues: 2.30000e-02 4.99900e-02 5.00000e-02 ... 3.56830e-01 4.45030e-01 4.74222e-01
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 10.967 seconds



Optimized geometry (Angstrom):
  Atom              X            Y            Z
  --------------------------------------------
  C          0.012017     0.013442     0.000000
  O          1.487176    -0.047359     0.000000
  H         -0.391894     1.044432    -0.000000
  H         -0.388934    -0.508279     0.895825
  H         -0.388934    -0.508279    -0.895825
  H          1.740677     0.946651     0.000000
converged SCF energy = -114.175513249242

Optimized B3LYP/STO-3G energy: -114.1755132492 Hartree
Energy lowering: -4.51 kcal/mol


## MinPop Implementation

The **Minimum Population Localization (MinPop)** method projects the Hartree-Fock
wavefunction from a large basis set onto a minimal basis (STO-3G) to obtain
well-conditioned Mulliken populations and bond orders.

**Key idea:** Standard Mulliken populations are basis-set dependent and can give
unphysical results with large basis sets. MinPop solves this by:

1. Computing the cross-overlap between STO-3G and the working basis
2. Projecting occupied MOs onto the minimal basis via:
   $C' = S'^{-1} \bar{S} C [C^T \bar{S}^T S'^{-1} \bar{S} C]^{-1/2}$
3. Localizing with Pipek-Mezey to get atom-centered orbitals
4. Computing Mulliken populations in this well-conditioned basis

*Reference: Montgomery, Frisch, Ochterski & Petersson, JCP 112, 6532 (2000)*

*Code adapted from Guido Falk von Rudorff: https://github.com/keithgroup/MinPop*


In [16]:
def calc_dm(C, Sbar, Sprime, Sprimeinv, minimal, O):
    """Compute MBS density matrix for one spin channel via MinPop.

    Parameters
    ----------
    C : ndarray — Occupied MO coefficients (large basis)
    Sbar : ndarray — Cross-overlap <minimal|large>
    Sprime : ndarray — Overlap matrix in the minimal basis
    Sprimeinv : ndarray — Inverse of Sprime
    minimal : pyscf.gto.Mole — Minimal basis molecule object
    O : ndarray — MO occupation numbers (> 0 only)
    """
    # MinPop projection
    P = Sbar @ C
    Cprime = P @ scipy.linalg.sqrtm(np.linalg.inv(C.T @ Sbar.T @ Sprimeinv @ P))
    Cprime = Sprimeinv @ Cprime

    # Pipek-Mezey localization (Mulliken charges)
    pm = pyscf.lo.PM(minimal, Cprime)
    pm.pop_method = "mulliken"
    loc_orb = pm.kernel()

    return (loc_orb * O) @ loc_orb.T


def run_minpop(calculation, spin=0):
    """Run MinPop analysis on a converged UHF calculation.

    Returns: (population, gross_pop, charges, occ_2s)
    """
    # Build minimal-basis molecule, preserving units from optimizer
    minimal = pyscf.gto.M(
        atom=calculation.mol.atom,
        basis="STO-3G",
        spin=spin,
        unit=calculation.mol.unit,
    )

    # Cross overlap: <STO-3G | working basis>
    Sbar = pyscf.gto.intor_cross("int1e_ovlp", minimal, calculation.mol)

    # Occupied MO coefficients and occupations (alpha, beta)
    C = [calculation.mo_coeff[i][:, calculation.mo_occ[i] > 0] for i in range(2)]
    O = [calculation.mo_occ[i][calculation.mo_occ[i] > 0] for i in range(2)]

    # Minimal-basis overlap and inverse
    Sprime = minimal.intor("int1e_ovlp")
    Sprimeinv = np.linalg.inv(Sprime)

    # MinPop density matrices
    dm_alpha = calc_dm(C[0], Sbar, Sprime, Sprimeinv, minimal, O[0])
    dm_beta  = calc_dm(C[1], Sbar, Sprime, Sprimeinv, minimal, O[1])

    # Full MBS Mulliken population analysis
    dm = dm_alpha + dm_beta
    pop = np.einsum("ij,ji->ij", dm, Sprime).real
    gross_pop = np.sum(pop, axis=0)

    # Condense to atoms
    population = np.zeros((minimal.natm, minimal.natm))
    for i, si in enumerate(minimal.ao_labels(fmt=None)):
        for j, sj in enumerate(minimal.ao_labels(fmt=None)):
            population[si[0], sj[0]] += pop[i, j]

    # Mulliken charges
    atomic_number = {'H':1, 'He':2, 'Li':3, 'Be':4, 'B':5,
                     'C':6, 'N':7, 'O':8, 'F':9}
    atoms = [calculation.mol.atom_symbol(i) for i in range(calculation.mol.natm)]
    charges = np.array([
        atomic_number[atoms[i]] - np.sum(population[:, i])
        for i in range(len(atoms))
    ])

    # 2s orbital occupations (needed by BEBOP1 for hybridization energy)
    occ_2s = []
    for i_ao, label in enumerate(minimal.ao_labels(fmt=None)):
        if label[2] == '2s':
            occ_2s.append(gross_pop[i_ao])
    occ_2s = np.array(occ_2s) if occ_2s else np.array([0.0])

    return population, gross_pop, charges, occ_2s

print("MinPop functions defined ✅")


MinPop functions defined ✅


## Step 2: MinPop with HF/6-31G*

Now we run an **Unrestricted Hartree-Fock** calculation with the 6-31G* basis,
then apply MinPop to get well-conditioned populations.

**Why UHF?** MinPop needs separate alpha and beta density matrices. For a
singlet, UHF gives identical alpha/beta orbitals (equivalent to RHF), but
the code needs the separated channels.

**Why `init_guess='huckel'`?** Closed-shell UHF can be tricky to converge.
The Hückel guess provides a better starting point than the default.


In [17]:
# Build molecule with 6-31G* basis using optimized geometry
# IMPORTANT: use unit=mol_eq.unit to preserve Bohr coordinates from optimizer
mol_631 = pyscf.gto.M(
    atom=mol_eq.atom,
    basis='6-31G*',
    charge=0, spin=0,
    unit=mol_eq.unit,
)
print(f"Basis: 6-31G* ({mol_631.nao} basis functions)")

# Run UHF
uhf_631 = pyscf.scf.UHF(mol_631)
uhf_631.max_cycle = 300
uhf_631.init_guess = 'huckel'
e_631 = uhf_631.kernel()
print(f"UHF/6-31G* energy: {e_631:.10f} Hartree")
print(f"Converged: {uhf_631.converged}")
print(f"<S²>: {uhf_631.spin_square()[0]:.2e} (should be ~0 for singlet)")


Basis: 6-31G* (36 basis functions)
converged SCF energy = -115.01983891067  <S^2> = 1.9802826e-10  2S+1 = 1
UHF/6-31G* energy: -115.0198389107 Hartree
Converged: True
<S²>: 1.98e-10 (should be ~0 for singlet)


In [18]:
# Run MinPop on the 6-31G* calculation
pop_631, gp_631, charges_631, occ2s_631 = run_minpop(uhf_631, spin=0)

atoms = [mol_eq.atom_symbol(i) for i in range(mol_eq.natm)]

print("MinPop Results — HF/6-31G*")
print("=" * 50)

print(f"\nMulliken Charges (e):")
for i, sym in enumerate(atoms):
    print(f"  {sym}({i+1}): {charges_631[i]:+.4f}")

print(f"\n2s Orbital Populations:")
hidx = 0
for i, sym in enumerate(atoms):
    if sym not in ('H', 'He'):
        print(f"  {sym}({i+1}): {occ2s_631[hidx]:.4f}")
        hidx += 1

print(f"\nCondensed-to-Atoms Population Matrix:")
n = len(atoms)
hdr = "        " + "".join(f"{sym}({i+1})    " for i, sym in enumerate(atoms))
print(hdr)
for i in range(n):
    row = f"{atoms[i]}({i+1})   "
    for j in range(n):
        row += f"{pop_631[i,j]:8.4f}"
    print(row)


MinPop Results — HF/6-31G*

Mulliken Charges (e):
  C(1): +0.0226
  O(2): -0.5049
  H(3): +0.0564
  H(4): +0.0705
  H(5): +0.0705
  H(6): +0.2848

2s Orbital Populations:
  C(1): 1.2303
  O(2): 1.8432

Condensed-to-Atoms Population Matrix:
        C(1)    O(2)    H(3)    H(4)    H(5)    H(6)    
C(1)     4.6346  0.2601  0.3729  0.3711  0.3711 -0.0325
O(2)     0.2601  8.0324 -0.0114 -0.0171 -0.0171  0.2582
H(3)     0.3729 -0.0114  0.6421 -0.0255 -0.0255 -0.0090
H(4)     0.3711 -0.0171 -0.0255  0.6262 -0.0273  0.0020
H(5)     0.3711 -0.0171 -0.0255 -0.0273  0.6262  0.0020
H(6)    -0.0325  0.2582 -0.0090  0.0020  0.0020  0.4944


## Step 3: MinPop with HF/cc-pVTZ

We repeat the same procedure with a **much larger** basis set (cc-pVTZ,
116 functions vs. 36 for 6-31G*). The key question: **do the MinPop bond
orders change?** If MinPop is working correctly, they should be nearly identical.


In [19]:
# Build molecule with cc-pVTZ basis
mol_pvtz = pyscf.gto.M(
    atom=mol_eq.atom,
    basis='cc-pVTZ',
    charge=0, spin=0,
    unit=mol_eq.unit,
)
print(f"Basis: cc-pVTZ ({mol_pvtz.nao} basis functions)")

# Run UHF
uhf_pvtz = pyscf.scf.UHF(mol_pvtz)
uhf_pvtz.max_cycle = 300
uhf_pvtz.init_guess = 'huckel'
e_pvtz = uhf_pvtz.kernel()
print(f"UHF/cc-pVTZ energy: {e_pvtz:.10f} Hartree")
print(f"Converged: {uhf_pvtz.converged}")


Basis: cc-pVTZ (116 basis functions)
converged SCF energy = -115.074821923383  <S^2> = 5.955556e-10  2S+1 = 1
UHF/cc-pVTZ energy: -115.0748219234 Hartree
Converged: True


In [20]:
# Run MinPop on cc-pVTZ
pop_pvtz, gp_pvtz, charges_pvtz, occ2s_pvtz = run_minpop(uhf_pvtz, spin=0)

print("MinPop Results — HF/cc-pVTZ")
print("=" * 50)

print(f"\nMulliken Charges (e):")
for i, sym in enumerate(atoms):
    print(f"  {sym}({i+1}): {charges_pvtz[i]:+.4f}")

print(f"\n2s Orbital Populations:")
hidx = 0
for i, sym in enumerate(atoms):
    if sym not in ('H', 'He'):
        print(f"  {sym}({i+1}): {occ2s_pvtz[hidx]:.4f}")
        hidx += 1

print(f"\nCondensed-to-Atoms Population Matrix:")
hdr = "        " + "".join(f"{sym}({i+1})    " for i, sym in enumerate(atoms))
print(hdr)
for i in range(n):
    row = f"{atoms[i]}({i+1})   "
    for j in range(n):
        row += f"{pop_pvtz[i,j]:8.4f}"
    print(row)


MinPop Results — HF/cc-pVTZ

Mulliken Charges (e):
  C(1): +0.0202
  O(2): -0.5367
  H(3): +0.0633
  H(4): +0.0740
  H(5): +0.0740
  H(6): +0.3052

2s Orbital Populations:
  C(1): 1.2299
  O(2): 1.8442

Condensed-to-Atoms Population Matrix:
        C(1)    O(2)    H(3)    H(4)    H(5)    H(6)    
C(1)     4.6364  0.2588  0.3735  0.3713  0.3713 -0.0314
O(2)     0.2588  8.0658 -0.0113 -0.0171 -0.0171  0.2577
H(3)     0.3735 -0.0113  0.6338 -0.0253 -0.0253 -0.0088
H(4)     0.3713 -0.0171 -0.0253  0.6222 -0.0271  0.0020
H(5)     0.3713 -0.0171 -0.0253 -0.0271  0.6222  0.0020
H(6)    -0.0314  0.2577 -0.0088  0.0020  0.0020  0.4733


## Sanity Check: PySCF vs. Gaussian MinPop

The table below compares our PySCF MinPop results against **Gaussian 16** ROHF MinPop
output for CH₃OH at the same geometry. The Gaussian route section used:

\# SP ROHF/<basis> Pop=(Full) IOp(6/27=122,6/12=3)

The relevant block in the Gaussian output is labeled
`MBS Condensed to atoms (all electrons):`.

Small differences are expected from: (1) UHF vs. ROHF and
(2) slightly different geometry from B3LYP VWN5 (PySCF) vs. VWN3 (Gaussian).
Bond orders — the quantities BEBOP1 actually uses — should agree to ~0.001.

Running Gaussian on the following input:

     # SP ROHF/cc-pVTZ Pop=(Full) IOp(6/27=122,6/12=3)

     CCSC Ethanol

     0 1
     C          0.012017     0.013442     0.000000
     O          1.487176    -0.047359     0.000000
     H         -0.391894     1.044432     0.000000
     H         -0.388934    -0.508279     0.895825
     H         -0.388934    -0.508279    -0.895825
     H          1.740677     0.946651     0.000000

Searching through the output for 'MBS' locates the following sections

     MBS Gross orbital populations:
                         Total     Alpha     Beta      Spin
     1 1   C  1S          1.99410   0.99705   0.99705   0.00000
     2        2S          1.22994   0.61497   0.61497   0.00000
     3        2PX         1.01332   0.50666   0.50666   0.00000
     4        2PY         0.71813   0.35907   0.35907   0.00000
     5        2PZ         1.02430   0.51215   0.51215   0.00000
     6 2   O  1S          1.99833   0.99916   0.99916   0.00000
     7        2S          1.84423   0.92211   0.92211   0.00000
     8        2PX         1.40842   0.70421   0.70421   0.00000
     9        2PY         1.31190   0.65595   0.65595   0.00000
    10        2PZ         1.97386   0.98693   0.98693   0.00000
    11 3   H  1S          0.93666   0.46833   0.46833   0.00000
    12 4   H  1S          0.92602   0.46301   0.46301   0.00000
    13 5   H  1S          0.92602   0.46301   0.46301   0.00000
    14 6   H  1S          0.69477   0.34739   0.34739   0.00000

     MBS Condensed to atoms (all electrons):
               1          2          3          4          5          6
     1  C    4.636398   0.258790   0.373510   0.371272   0.371272  -0.031447
     2  O    0.258790   8.065795  -0.011315  -0.017097  -0.017097   0.257669
     3  H    0.373510  -0.011315   0.633751  -0.025261  -0.025261  -0.008765
     4  H    0.371272  -0.017097  -0.025261   0.622199  -0.027087   0.001990
     5  H    0.371272  -0.017097  -0.025261  -0.027087   0.622199   0.001990
     6  H   -0.031447   0.257669  -0.008765   0.001990   0.001990   0.473334

     MBS Mulliken charges and spin densities:
               1          2
     1  C    0.020206   0.000000
     2  O   -0.536744   0.000000
     3  H    0.063340   0.000000
     4  H    0.073985   0.000000
     5  H    0.073985   0.000000
     6  H    0.305228   0.000000



## Basis Set Comparison

This result demonstrates **MinPop's basis-set sensitivity** and why Mulliken bond orders would be reliable AFTER projecting to a minimum basis set and projection. Despite going from 36 to 116 basis functions (a 3× increase), the bond orders for CH3OH hardly change at all, but be cautious using too small a basis set.  Sensitivities pertaining to geometries, basis sets, levels of theory are all not well-established yet.  


In [21]:
print("Basis Set Comparison: MinPop Bond Orders")
print("=" * 55)
print(f"\n  {'Bond':<18} {'6-31G*':>10} {'cc-pVTZ':>10} {'Diff':>10}")
print(f"  {'-'*50}")

for i in range(1, len(atoms)):
    for j in range(i):
        b1, b2 = pop_631[i, j], pop_pvtz[i, j]
        if abs(b1) > 0.05 or abs(b2) > 0.05:
            bond = f"{atoms[j]}({j+1})-{atoms[i]}({i+1})"
            print(f"  {bond:<18} {b1:10.4f} {b2:10.4f} {b2-b1:10.4f}")

print(f"\n  HF Energies:")
print(f"    HF/6-31G*:   {e_631:.10f} Ha")
print(f"    HF/cc-pVTZ:  {e_pvtz:.10f} Ha")
print(f"    Difference:  {(e_pvtz - e_631)*627.509:.2f} kcal/mol")
print(f"\n  Note: While the HF energies differ by ~{(e_pvtz - e_631)*627.509:.0f} kcal/mol,")
print(f"  the MinPop bond orders agree to ~0.001. This is the key feature we can exploit!")


Basis Set Comparison: MinPop Bond Orders

  Bond                   6-31G*    cc-pVTZ       Diff
  --------------------------------------------------
  C(1)-O(2)              0.2601     0.2588    -0.0013
  C(1)-H(3)              0.3729     0.3735     0.0006
  C(1)-H(4)              0.3711     0.3713     0.0001
  C(1)-H(5)              0.3711     0.3713     0.0001
  O(2)-H(6)              0.2582     0.2577    -0.0005

  HF Energies:
    HF/6-31G*:   -115.0198389107 Ha
    HF/cc-pVTZ:  -115.0748219234 Ha
    Difference:  -34.50 kcal/mol

  Note: While the HF energies differ by ~-35 kcal/mol,
  the MinPop bond orders agree to ~0.001. This is the key feature we can exploit!


## Step 4: BEBOP1 Bond Energy Calculation

**BEBOP1** (Bond Energy / Bond Order Population) converts the MinPop
populations into molecular atomization energies and individual bond energies
using parameters fitted to CBS-QB3 benchmark data.

The BEBOP1 energy has three components:
1. **Covalent bond energy**: $E_{cov} = -2\beta_{AB} \cdot P_{AB}$ (from bond orders)
2. **Short-range repulsion**: $E_{rep} = D_{AB} \exp[\zeta_{AB}(R - R_\sigma)]$
3. **Hybridization energy**: $(n_{2s}^{ref} - n_{2s}^{mol}) \cdot E_{2s \to 2p}$

*Note: BEBOP1 parameters were fitted to ROHF/CBSB3 (6-311+G(3d2f,2df,2p)). We use HF/6-31G\*
MinPop data as the a much less expensive example.*


In [13]:
# Compute distance matrix from optimized geometry (Angstrom)
atoms_arr = np.array(atom_symbols)
DistMat, Angles = sg.Spatial_Properties(opt_coords_ang)

print("Key bond distances (Angstrom):")
for i in range(len(atoms_arr)):
    for j in range(i+1, len(atoms_arr)):
        d = DistMat[i][j]
        if d < 1.6:  # bonded pairs only
            print(f"  {atoms_arr[i]}({i+1})-{atoms_arr[j]}({j+1}): {d:.4f}")


Key bond distances (Angstrom):
  C(1)-O(2): 1.4764
  C(1)-H(3): 1.1073
  C(1)-H(4): 1.1115
  C(1)-H(5): 1.1115
  O(2)-H(6): 1.0258


In [14]:
# Compute BEBOP1 total atomization energy
total_E = bebop_eq.BEBOP1(DistMat, atoms_arr, occ2s_631, pop_631)
print(f"BEBOP1 Total Atomization Energy (incl. ZPVE): {total_E:.2f} kcal/mol")

# Compute individual bond energies
data = bebop_eq.BEBOPBondEnergies(
    pop_631, pop_631, pop_631, DistMat, atoms_arr, occ2s_631
)
Esig, Epi, Ecov, Enet_sig, Enet_pi, Enet, CompTable, BEBOP_tot = data

print(f"\nIndividual Bond Energies (kcal/mol):")
print(f"  {'Bond':<18} {'Net E':>10} {'Gross E':>10}")
print(f"  {'-'*40}")
for i in range(1, len(atoms_arr)):
    for j in range(i):
        if abs(Enet[i][j]) > 5.0:
            bond = f"{atoms_arr[j]}({j+1})-{atoms_arr[i]}({i+1})"
            print(f"  {bond:<18} {Enet[i][j]:10.2f} {Ecov[i][j]:10.2f}")

print(f"\n  Total BEBOP1 energy: {BEBOP_tot:.2f} kcal/mol")

print(f"\nHybridization Energies (2s -> 2p promotion):")
for i, sym in enumerate(atoms_arr):
    if CompTable[i][i] != 0:
        print(f"  {sym}({i+1}): {CompTable[i][i]:+.2f} kcal/mol")


BEBOP1 Total Atomization Energy (incl. ZPVE): -486.03 kcal/mol

Individual Bond Energies (kcal/mol):
  Bond                    Net E    Gross E
  ----------------------------------------
  C(1)-O(2)             -111.54    -149.68
  C(1)-H(3)             -105.41    -123.78
  O(2)-H(3)                5.26       5.89
  C(1)-H(4)             -105.09    -123.41
  O(2)-H(4)                7.91       8.86
  H(3)-H(4)                7.39       7.39
  C(1)-H(5)             -105.09    -123.41
  O(2)-H(5)                7.91       8.86
  H(3)-H(5)                7.39       7.39
  H(4)-H(5)                7.90       7.90
  C(1)-H(6)                9.88      11.61
  O(2)-H(6)             -113.99    -127.56

  Total BEBOP1 energy: -486.03 kcal/mol

Hybridization Energies (2s -> 2p promotion):
  C(1): +75.50 kcal/mol
  O(2): +26.99 kcal/mol


Here is preferred output from our local BEBOP1 code run on the Gaussian output following instructions on GitHub:


                    SUMMARY OF BEBOP-1 CALCULATION

                        BEBOP-1 (Version 1.0.1)
                       10-March-2026 20:25:09


     HARTREE-FOCK OUTPUT:  /ihome/jkeith/jakeith/BEBOP1_github/CCSC_TZ_minpop.out


     BEBOP ATOMIZATION ENERGY (0 K)     =     -486.17070619752224 KCAL/MOL


     GROSS SIGMA BOND ENERGIES INCLUDING REPULSION
                   1          2          3          4          5
     1     C     0.00
     2     O  -148.93       0.00
     3     H  -123.98       5.85       0.00
     4     H  -123.46       8.84       7.32       0.00
     5     H  -123.46       8.84       7.32       7.84       0.00
     6     H    11.24    -127.31       2.54      -0.58      -0.58
                   6
     6     H     0.00


     TOTAL GROSS SIGMA ENERGY     =     -588.51 KCAL/MOL


     GROSS PI BOND ENERGIES INCLUDING REPULSION
                   1          2          3          4          5
     1     C     0.00
     2     O     0.00       0.00
     3     H     0.00       0.00       0.00
     4     H     0.00       0.00       0.00       0.00
     5     H     0.00       0.00       0.00       0.00       0.00
     6     H     0.00       0.00       0.00       0.00       0.00
                   6
     6     H     0.00


     TOTAL GROSS PI  ENERGY     =     0.00 KCAL/MOL


     GROSS TOTAL BOND ENERGIES INCLUDING REPULSION
                   1          2          3          4          5
     1     C     0.00
     2     O  -148.93       0.00
     3     H  -123.98       5.85       0.00
     4     H  -123.46       8.84       7.32       0.00
     5     H  -123.46       8.84       7.32       7.84       0.00
     6     H    11.24    -127.31       2.54      -0.58      -0.58
                   6
     6     H     0.00


     TOTAL GROSS ENERGY     =     -588.51 KCAL/MOL


     NET SIGMA BOND ENERGIES INCLUDING HYBRIDIZATION
                   1          2          3          4          5
     1     C     0.00
     2     O  -111.01       0.00
     3     H  -105.57       5.23       0.00
     4     H  -105.12       7.90       7.32       0.00
     5     H  -105.12       7.90       7.32       7.84       0.00
     6     H     9.57    -113.81       2.54      -0.58      -0.58
                   6
     6     H     0.00


     TOTAL NET SIGMA ENERGY     =     -486.17 KCAL/MOL


     NET PI BOND ENERGIES INCLUDING HYBRIDIZATION
                   1          2          3          4          5
     1     C     0.00
     2     O     0.00       0.00
     3     H     0.00       0.00       0.00
     4     H     0.00       0.00       0.00       0.00
     5     H     0.00       0.00       0.00       0.00       0.00
     6     H     0.00       0.00       0.00       0.00       0.00
                   6
     6     H     0.00


     TOTAL NET PI BOND ENERGY     =     0.00 KCAL/MOL


     NET TOTAL ENERGIES INCLUDING HYBRIDIZATION
                   1          2          3          4          5
     1     C     0.00
     2     O  -111.01       0.00
     3     H  -105.57       5.23       0.00
     4     H  -105.12       7.90       7.32       0.00
     5     H  -105.12       7.90       7.32       7.84       0.00
     6     H     9.57    -113.81       2.54      -0.58      -0.58
                   6
     6     H     0.00


     TOTAL NET ENERGY     =     -486.17 KCAL/MOL


     COMPOSITE TABLE:    EII(HYBRID)  EIJ(GROSS)
                         EJI(NET)     EJJ(HYBRID)
                   1          2          3          4          5
     1     C    75.54       0.00       0.00       0.00       0.00
     2     O  -111.01      26.80       0.00       0.00       0.00
     3     H  -105.57       5.23       0.00       0.00       0.00
     4     H  -105.12       7.90       7.32       0.00       0.00
     5     H  -105.12       7.90       7.32       7.84       0.00
     6     H     9.57    -113.81       2.54      -0.58      -0.58
                   6
     1     C     0.00
     2     O     0.00
     3     H     0.00
     4     H     0.00
     5     H     0.00
     6     H     0.00


     TOTAL HYBRIDIZATION ENERGY     =     102.34 KCAL/MOL


     SORTED NET BONDING ENERGIES (LOWEST TO HIGHEST)

     ----------------------------
     BOND           BOND ENERGIES
     IDENTITY       (KCAL/MOL)  
     ----------------------------
      O2-H6         -113.81
      C1-O2         -111.01
      C1-H3         -105.57
      C1-H4         -105.12
      C1-H5         -105.12
     ----------------------------



## Summary & Discussion Points

### Key Results

1. **MinPop basis-set independence**: Bond orders from 6-31G* and cc-pVTZ
   agree to ~0.001, despite a 3× difference in basis set size. This is the
   central advantage of the MinPop approach over standard Mulliken analysis. BEBOP1 has parameters fit assuming the use MinPop with ROHF/6-311G(2d,d,p) calculations to coincide with the CBS-QB3 method.  

2. **Hybridization energy**: Carbon's 2s→2p promotion costs ~75 kcal/mol,
   while oxygen's is ~27 kcal/mol. These are real physical costs that BEBOP1
   accounts for that bring chemical insight.

3. **Net vs. Gross bond energies**: Gross energies include only covalent +
   repulsion terms. Net energies additionally distribute hybridization
   corrections among the bonds.

### Workshop Exercises

Try modifying this notebook to explore:

- **Different molecules**: Replace CH₃OH with H₂O, CH₄, C₂H₆, or CH₂O
- **Different basis sets**: Try 6-311G**, aug-cc-pVDZ, or def2-TZVP
- **Open-shell systems**: Compare bond energies and hybridization energies of CH₂ singlet and triplet.  Change `spin=2` for a triplet (e.g., CH₂)
  — see the `CH2_triplet.py` example in the MinPop repo

### References

- **BEBOP-1 paper**: Zulueta et al., *JCTC* 2022, 18, 4774–4794
- **MinPop method**: Montgomery, Frisch, Ochterski & Petersson, *JCP* 2000, 112, 6532
- **PySCF**: Sun et al., *WIREs Comput. Mol. Sci.* 2018, 8, e1340
